# Parameterized sonar image registration pipeline

This notebook keeps the original XTF → section extraction → PhysDNet → mask → matching → UTM conversion → custom 3-DOF RANSAC flow.

The main addition is the `MATCHER_BACKEND` parameter:

- `"minima_sp_lg"`: current MINIMA + LightGlue + SuperPoint pipeline.
- `"lightglue_superpoint"`: direct SuperPoint + LightGlue from the cloned `cvg/LightGlue` repository.
- `"sift"`: OpenCV SIFT feature extraction + descriptor matching.
- `"lightglue_sift"`: optional SIFT + LightGlue through the LightGlue repository.

All matcher outputs are normalized to the same format: `mkpts0`, `mkpts1`, and `mconf`. After that, the same mask filtering, pixel-to-UTM conversion, and custom RANSAC are used.

Left-side orientation rule included in the companion scripts:

- the final left mask is built from shadow + height/z + blind-zone masks in native orientation, then the merged final mask is flipped once;
- every left image source is flipped horizontally before matching, including raw sonar and PhysDNet reflectance;
- the left swath grid is flipped horizontally before pixel-to-UTM conversion;
- the notebook now displays the exact final masks and overlays that will be applied before the matcher runs.

- every selected left mask component is flipped horizontally before final-mask combination;
- every left image source is flipped horizontally before matching, including raw sonar and PhysDNet reflectance;
- the left swath grid is flipped horizontally before pixel-to-UTM conversion;
- this applies regardless of `MATCH_MODE`: `left-left`, `left-right`, `right-left`, `right-right`, or `both`.


# 1) Imports and global configuration

In [ ]:
import os
import sys
from pathlib import Path
from typing import Optional

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

from prep import *
from inference import *
from xtf_utils import *
from masks import *

# Existing helper file from your current pipeline.
# It builds the matching inputs, applies masks, converts pixels to UTM,
# and calls your custom 3-DOF RANSAC implementation.
from minima_pipeline import *

torch.set_grad_enabled(False)

In [ ]:
# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------
xtf_file = r"2025-09-24_09-25-24_0.xtf"
xtf_dir = r"/home/david/Documents/IFRoS/Perception/HoP-SSS-Loop-Closure/data"

output_dir = r"/home/david/Documents/IFRoS/Perception/HoP-SSS-Loop-Closure/output_simplified"

# PhysDNet weights
# weight_path = r"/home/david/Documents/IFRoS/Perception/HoP-SSS-Loop-Closure/best_model_v610_eagle.pth"
weight_path = r"/home/david/Documents/IFRoS/Perception/HoP-SSS-Loop-Closure/best_model_v610_jaguar.pth"
# weight_path = r"/home/david/Documents/IFRoS/Perception/HoP-SSS-Loop-Closure/best_model_v611_jaguar.pth"

# If LightGlue was cloned but not installed with `pip install -e .`, set this path.
# Example: LIGHTGLUE_REPO = r"/home/david/Documents/IFRoS/Perception/LightGlue"
LIGHTGLUE_REPO = r""
if LIGHTGLUE_REPO and LIGHTGLUE_REPO not in sys.path:
    sys.path.insert(0, LIGHTGLUE_REPO)

# MINIMA checkpoint
MINIMA_CKPT = r"MINIMA/weights/minima_lightglue.pth"

# -------------------------------------------------------------------------
# Device
# -------------------------------------------------------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------------------------------
# XTF section extraction parameters
# -------------------------------------------------------------------------
segment_size = 2000
upper_limit = 2 ** 15

# FAILED
# timestamp1 = 48200
# timestamp2 = 46150

# WEIRD
# timestamp1 = 13700
# timestamp2 = 11300

timestamp1 = 52200
timestamp2 = 48450

# timestamp2 = 30000
# timestamp1 = 31000

# timestamp1 = 100
# timestamp2 = 110

# Easy flat surface
# timestamp1 = 4750
# timestamp2 = 1250

# -------------------------------------------------------------------------
# Terrain/mask parameters
# -------------------------------------------------------------------------
TERRAIN_MASK_K = 1.0

# Choose which masks are combined into terrain.final_mask.
# Available names in the code below: "z", "shadow", "blind".
# Use all rejection masks by default. You can still reduce this list for ablation tests.
MASK_COMPONENTS = ["shadow", "blind"]
# MASK_COMPONENTS = ["blind"]

# Horizontal orientation of the final LEFT mask.
# False means: merge shadow/z/blind and DO NOT flip the final left mask.
# True means: merge shadow/z/blind and then flip the final left mask once.
# Use the debug overlay cell below to choose the value that aligns the white rejection
# regions with the image actually passed to the matcher.
FLIP_LEFT_FINAL_MASK = False

# -------------------------------------------------------------------------
# Matching input parameters
# -------------------------------------------------------------------------
MATCH_MODE = "both"
# MATCH_MODE = "left-left"
# MATCH_MODE = "right-right"
# MATCH_MODE = "left-right"
# MATCH_MODE = "right-left"

IMAGE_SOURCE = "rho_gray"
# IMAGE_SOURCE = "raw"

# Flip every LEFT input image before concatenation/matching.
# This applies to both IMAGE_SOURCE="raw" and IMAGE_SOURCE="rho_gray".
# Keep this consistent with the final left mask orientation shown in the debug overlay.
FLIP_LEFT_INPUT_IMAGE = False

APPLY_MASK = True

# -------------------------------------------------------------------------
# Matcher backend
# -------------------------------------------------------------------------
# MATCHER_BACKEND = "minima_sp_lg"
# MATCHER_BACKEND = "lightglue_superpoint"
MATCHER_BACKEND = "sift"
# MATCHER_BACKEND = "lightglue_sift"   # optional

# Direct LightGlue parameters.
LIGHTGLUE_MAX_NUM_KEYPOINTS = 4096
LIGHTGLUE_RESIZE = None                 # None disables auto-resize. Use an int or (h, w) if needed.
LIGHTGLUE_DEPTH_CONFIDENCE = 0.95       # Lower is faster; -1 disables early stopping.
LIGHTGLUE_WIDTH_CONFIDENCE = 0.99       # Lower is faster; -1 disables point pruning.
LIGHTGLUE_FILTER_THRESHOLD = 0.1        # Higher gives fewer but stronger matches.
LIGHTGLUE_MP = False                    # Mixed precision.
LIGHTGLUE_FLASH = True
LIGHTGLUE_COMPILE = False

# OpenCV SIFT parameters.
SIFT_NFEATURES = 8000
SIFT_N_OCTAVE_LAYERS = 3
SIFT_CONTRAST_THRESHOLD = 0.01
SIFT_EDGE_THRESHOLD = 10
SIFT_SIGMA = 1.6
SIFT_MATCHER = "flann"                 # "flann" or "bf"
SIFT_RATIO_TEST = 0.75
SIFT_CROSS_CHECK = False               # Only used when SIFT_MATCHER="bf".
SIFT_FLANN_TREES = 5
SIFT_FLANN_CHECKS = 64
SIFT_ENFORCE_UNIQUE_MATCHES = True
SIFT_MAX_MATCHES = None                # Example: 2000, or None to keep all surviving matches.

# -------------------------------------------------------------------------
# RANSAC parameters
# -------------------------------------------------------------------------
RANSAC_REPROJ_THRESHOLD_PIXEL = 8.0      # pixels, used for H_pixel
RANSAC_REPROJ_THRESHOLD_UTM = 0.3       # metres, used for H_utm
RANSAC_MAX_ITERS = 10_000
RANSAC_CONFIDENCE = 0.95
RANSAC_OUTLIER_PERCENT = 0.6

MAX_DRAW_MATCHES = 300

# Optional debug self-rotation check.
RUN_DEBUG_SELF_ROTATION = False

# 2) Display data and validate timestamps

In [ ]:
def display_figure(images_dict: dict, lower: int, upper: int, main_title: str):
    images = list(images_dict.values())
    titles = list(images_dict.keys())
    n = len(images)

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

    if n == 1:
        axes = [axes]

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        ax.set_title(title)

    plt.suptitle(f"{main_title}: {lower} to {upper}")
    plt.tight_layout()
    plt.show()

In [ ]:
data_load = load_xtf(xtf_dir + r"/" + xtf_file)
swaths, trajectory, altitude, roll, pitch, yaw = calculate_swath_positions(data_load)

_, ax = plt.subplots(figsize=(10, 5))

# Plot trajectory
ax.plot(trajectory[:, 0], trajectory[:, 1], color="blue", label="trajectory")

# Plot ping lines
step = 50
for i in range(0, swaths.shape[0], step):
    start_point = swaths[i, 0, :]
    end_point = swaths[i, -1, :]

    x_vals = [start_point[0], end_point[0]]
    y_vals = [start_point[1], end_point[1]]

    if i == 0:
        ax.plot(x_vals, y_vals, color="green", linestyle="-", alpha=0.4, linewidth=1, label=f"swaths with step={step}")
    else:
        ax.plot(x_vals, y_vals, color="green", linestyle="-", alpha=0.4, linewidth=1)

ax.axis("equal")
ax.set_title("Trajectory")
ax.set_xlabel("Easting [metres]")
ax.set_ylabel("Northing [metres]")
ax.scatter([trajectory[0, 0]], [trajectory[0, 1]], label="Start", c="red")
ax.scatter([trajectory[-1, 0]], [trajectory[-1, 1]], label="End", c="green")
ax.legend()
plt.show()

filter, dy = lowpass_and_derivative_5point(
    np.arange(len(np.unwrap(yaw) * 180 / np.pi)),
    np.unwrap(yaw) * 180 / np.pi,
    0.01,
)

# Threshold the derivative to find where 
threshold = 0.08
Ids = np.where(np.abs(dy)>threshold)[0]
new_index = range(0, swaths.shape[0],1)
new_index = np.delete(new_index,Ids)

# Detect different groups after removing turns
changes = 1
jumps = np.where(np.diff(new_index) > changes)[0] + 1
groups = np.split(new_index, jumps)

# Preserve only large groups
min_group_size = 29
groups_indices = [g for g in groups if len(g) > min_group_size]

print("Count of groups: ", len(groups_indices))
max_values = [np.max(i) for i in groups_indices]

# Plot the trajectory with different colors for each group
_, ax = plt.subplots(figsize=(10,5))
for i in range(len(groups_indices)):
    minimum = min(groups_indices[i])
    maximum = max(groups_indices[i])
    ax.plot(trajectory[minimum:maximum,0], trajectory[minimum:maximum,1])

ax.axis("equal")
ax.set_title("Trajectory")
ax.set_xlabel("Easting [metres]")
ax.set_ylabel("Northing [metres]")
ax.axis("equal")
ax.scatter([trajectory[0,0]], [trajectory[0,1]], label="Start", c="red")
ax.scatter([trajectory[-1,0]], [trajectory[-1,1]], label="End", c="green")
ax.scatter(trajectory[timestamp1,0], trajectory[timestamp1,1], label="Points of Interest", c="orange")
ax.scatter(trajectory[timestamp2,0], trajectory[timestamp2,1], label="Points of Interest", c="orange")
ax.legend()

plt.show()

print(f"Coordinates of timestamp 1: {trajectory[timestamp1,0]}, {trajectory[timestamp1,1]}")
print(f"Coordinates of timestamp 2: {trajectory[timestamp2,0]}, {trajectory[timestamp2,1]}")
dx = trajectory[timestamp2,0] - trajectory[timestamp1,0]
dy = trajectory[timestamp2,1] - trajectory[timestamp1,1]
print(f"Relative displacement: {dx}, {dy}")

In [ ]:
t1_group_idx: bool = None
t2_group_idx: bool = None

# Check all the groups to identify the indices.
for i, g in enumerate(groups_indices):
    if timestamp1 in g:
        t1_group_idx = i
    if timestamp2 in g:
        t2_group_idx = i

if t1_group_idx is not None and t2_group_idx is not None:
    print(
        f"Both timestamp1={timestamp1} and timestamp2={timestamp2} are valid. "
        f"They belong to groups {t1_group_idx} and {t2_group_idx}."
    )
elif t1_group_idx is not None and t2_group_idx is None:
    print(f"WARNING: timestamp1={timestamp1} is valid, but timestamp2={timestamp2} corresponds to a turn.")
elif t1_group_idx is None and t2_group_idx is not None:
    print(f"WARNING: timestamp1={timestamp1} corresponds to a turn, but timestamp2={timestamp2} is valid.")
else:
    print(f"WARNING: Neither timestamp1={timestamp1} nor timestamp2={timestamp2} is valid; both correspond to turns.")

# 3) Get the indices for the desired range

In [ ]:
# The range of the window should be compatible with PhysDNet.
# Ideally, the timestamp should be in the middle of the window, but if it is too close
# to the beginning or end of the group, it will be adjusted by get_range_window().

group1 = groups_indices[t1_group_idx]
group2 = groups_indices[t2_group_idx]

lower1, upper1 = get_range_window(group1, timestamp1)
lower2, upper2 = get_range_window(group2, timestamp2)

print("Section 1:", lower1, upper1, "size:", upper1 - lower1)
print("Section 2:", lower2, upper2, "size:", upper2 - lower2)

# 4) Prepare data for PhysDNet

## Section 1

In [ ]:
section1: SonarSectionData = prepare_data_range(
    input_dir        = xtf_dir,
    xtf_file         = xtf_file,
    segment_size     = segment_size,
    upper_limit      = upper_limit,
    side             = "both",
    data_lower_limit = lower1,
    data_upper_limit = upper1,
)

save_sonar_section(section1, output_dir=output_dir + r"/section_01", xtf_file=xtf_file)

display_figure(section1.images, lower1, upper1, "Images for section 1")
display_figure(section1.shadows, lower1, upper1, "Shadows for section 1")

## Section 2

In [ ]:
section2: SonarSectionData = prepare_data_range(
    input_dir        = xtf_dir,
    xtf_file         = xtf_file,
    segment_size     = segment_size,
    upper_limit      = upper_limit,
    side             = "both",
    data_lower_limit = lower2,
    data_upper_limit = upper2,
)

save_sonar_section(section2, output_dir=output_dir + r"/section_02", xtf_file=xtf_file)

display_figure(section2.images, lower2, upper2, "Images for section 2")
display_figure(section2.shadows, lower2, upper2, "Shadows for section 2")

# 5) Run PhysDNet inference

In [ ]:
inference1 = run_inference(
    section     = section1,
    weight_path = weight_path,
    device      = device,
)

save_inference_result(inference1, output_dir + r"/section_01/inference")
display_figure(inference1.rho_gray, lower1, upper1, "Rho gray for section 1")

In [ ]:
inference2 = run_inference(
    section     = section2,
    weight_path = weight_path,
    device      = device,
)

save_inference_result(inference2, output_dir + r"/section_02/inference")
display_figure(inference2.rho_gray, lower2, upper2, "Rho gray for section 2")

# 6) Get masks from terrain and height

## Section 1

In [ ]:
terrain1 = terrain_mask(
    section   = section1,
    inference = inference1,
    k         = TERRAIN_MASK_K,
)

save_terrain_result(terrain1, output_dir + r"/section_01")
display_figure(terrain1.z_mask, lower1, upper1, "Height mask for section 1")

## Section 2

In [ ]:
terrain2 = terrain_mask(
    section   = section2,
    inference = inference2,
    k         = TERRAIN_MASK_K,
)

save_terrain_result(terrain2, output_dir + r"/section_02")
display_figure(terrain2.z_mask, lower2, upper2, "Height mask for section 2")

# 7) Combine masks

`terrain.final_mask` is the mask used after matching. Black pixels are kept and white pixels are rejected.

The robust orientation rule is: each component mask is created in the native raw-image orientation; the selected components are merged; then the final left mask is flipped once so it matches the flipped left raw image / reflectance map used by the matcher. This avoids a partial-flip situation where only the blind mask is mirrored.


In [ ]:
def build_final_masks(section, terrain, output_mask_dir: str, lower: int, upper: int, title: str):
    """
    Builds terrain.final_mask from the selected mask components.

    Mask convention:
        0     -> valid / keep
        > 0   -> invalid / reject

    Orientation rule:
        - shadow, z/height, and blind masks are merged in their native orientation.
        - the final LEFT mask is flipped only if FLIP_LEFT_FINAL_MASK=True.
        - the image flip is controlled separately by FLIP_LEFT_INPUT_IMAGE.
    """
    os.makedirs(output_mask_dir, exist_ok=True)

    # Avoid stale masks from previous notebook executions.
    terrain.final_mask = {}

    for side in ["left", "right"]:
        shadow_key = get_key_by_side(section.shadows, side)
        z_key = get_key_by_side(terrain.z_mask, side)
        blind_key = get_key_by_side(section.blind, side)

        components_by_name = {
            "shadow": section.shadows[shadow_key],
            "z": terrain.z_mask[z_key],
            "blind": section.blind[blind_key],
        }

        selected_components = {}
        selected_masks = []

        for name in MASK_COMPONENTS:
            if name not in components_by_name:
                raise ValueError(
                    f"Invalid mask component '{name}'. "
                    f"Use one of: {list(components_by_name.keys())}"
                )

            selected_components[name] = components_by_name[name]
            selected_masks.append(components_by_name[name])

        final_key = f"{side}_final_mask"
        terrain.final_mask[final_key] = combine_masks_for_matching(
            selected_masks,
            side=side,
            flip_left_final=FLIP_LEFT_FINAL_MASK,
        )

        cv2.imwrite(
            os.path.join(output_mask_dir, f"{final_key}.png"),
            terrain.final_mask[final_key],
        )

        show_mask_component_debug(
            section=section,
            terrain=terrain,
            side=side,
            selected_components=selected_components,
            final_mask=terrain.final_mask[final_key],
            title=f"{title} - {side} mask components and final applied mask",
        )

    display_figure(terrain.final_mask, lower, upper, title)
    
def combine_masks_for_matching(
    masks: list[np.ndarray],
    side: str,
    flip_left_final: bool = False,
) -> np.ndarray:
    """
    Build the exact final rejection mask used by the matcher.

    Mask convention:
        0     -> valid / keep
        > 0   -> invalid / reject

    The safe rule is:
        1. Keep all selected mask components in the orientation produced by the
           preprocessing/inference code.
        2. Merge them into one final rejection mask.
        3. Optionally flip only the final LEFT mask, controlled by
           FLIP_LEFT_FINAL_MASK.
    """
    if not masks:
        raise ValueError("masks list is empty")

    side = side.lower().strip()
    if side not in {"left", "right"}:
        raise ValueError(f"Invalid side='{side}'. Use 'left' or 'right'.")

    target_shape = np.asarray(masks[0]).shape[:2]
    normalised = []

    for mask in masks:
        mask = np.asarray(mask)

        if mask.ndim == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        if mask.shape[:2] != target_shape:
            mask = cv2.resize(
                mask,
                (target_shape[1], target_shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        normalised.append(mask)

    final_mask = (sum(mask > 0 for mask in normalised) > 0).astype(np.uint8) * 255

    if flip_left_final and side == "left":
        final_mask = np.fliplr(final_mask)

    return np.ascontiguousarray(final_mask)


def show_mask_component_debug(section, terrain, side: str, selected_components: dict, final_mask: np.ndarray, title: str):
    """
    Shows the selected mask components and the final mask that will be stored in
    terrain.final_mask.
    """
    n_components = len(selected_components)
    fig, axes = plt.subplots(1, n_components + 1, figsize=(5 * (n_components + 1), 5))

    if n_components + 1 == 1:
        axes = [axes]

    for ax, (name, mask) in zip(axes[:-1], selected_components.items()):
        ax.imshow(mask, cmap="gray")
        ax.set_title(f"{side} component: {name}")
        ax.axis("off")

    axes[-1].imshow(final_mask, cmap="gray")
    flip_state = "flipped" if (side == "left" and FLIP_LEFT_FINAL_MASK) else "not flipped"
    axes[-1].set_title(f"{side} FINAL applied mask ({flip_state})")
    axes[-1].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def prepare_side_image_and_mask(
    inference_obj,
    terrain_obj,
    side: str,
    image_source: str = "rho_gray",
    section_obj=None,
    flip_left_image: bool | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns one side image and its corresponding final mask in matching orientation.

    This local definition intentionally overrides the imported minima_pipeline version
    so notebook changes are not affected by stale imports.

    Orientation rule:
        - LEFT image is flipped when FLIP_LEFT_INPUT_IMAGE=True.
        - Mask orientation is controlled only by FLIP_LEFT_FINAL_MASK during
          build_final_masks(); it is not flipped again here.
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    side = side.lower().strip()
    if side not in {"left", "right"}:
        raise ValueError(f"Invalid side='{side}'. Use 'left' or 'right'.")

    image_dict = get_image_dict_from_source(
        image_source=image_source,
        inference_obj=inference_obj,
        section_obj=section_obj,
    )

    img_key = get_key_by_side(image_dict, side)
    mask_key = get_key_by_side(terrain_obj.final_mask, side)

    img = image_dict[img_key]
    mask = terrain_obj.final_mask[mask_key]

    if side == "left" and flip_left_image:
        img = np.fliplr(img)

    img = ensure_uint8_image(img)
    img = np.ascontiguousarray(img)

    mask = resize_mask_to_image(mask, img)
    mask = np.ascontiguousarray(mask)

    return img, mask


def build_matching_inputs(
    mode: str,
    inference0,
    inference1,
    terrain0,
    terrain1,
    image_source: str = "rho_gray",
    section0=None,
    section1=None,
    flip_left_image: bool | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, str]:
    """
    Builds the exact images and masks used for matching.

    For MATCH_MODE="both", the convention is always:
        image = [LEFT_in_matching_orientation | RIGHT_native]
        mask  = [LEFT_final_mask             | RIGHT_final_mask]
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    mode = mode.lower().strip()

    if mode == "both":
        left0_img, left0_mask = prepare_side_image_and_mask(
            inference0,
            terrain0,
            "left",
            image_source=image_source,
            section_obj=section0,
            flip_left_image=flip_left_image,
        )
        right0_img, right0_mask = prepare_side_image_and_mask(
            inference0,
            terrain0,
            "right",
            image_source=image_source,
            section_obj=section0,
            flip_left_image=flip_left_image,
        )

        left1_img, left1_mask = prepare_side_image_and_mask(
            inference1,
            terrain1,
            "left",
            image_source=image_source,
            section_obj=section1,
            flip_left_image=flip_left_image,
        )
        right1_img, right1_mask = prepare_side_image_and_mask(
            inference1,
            terrain1,
            "right",
            image_source=image_source,
            section_obj=section1,
            flip_left_image=flip_left_image,
        )

        img0 = np.ascontiguousarray(np.hstack([left0_img, right0_img]))
        img1 = np.ascontiguousarray(np.hstack([left1_img, right1_img]))

        mask0 = np.ascontiguousarray(np.hstack([left0_mask, right0_mask]))
        mask1 = np.ascontiguousarray(np.hstack([left1_mask, right1_mask]))

        return img0, img1, mask0, mask1, "both", "both"

    valid_modes = {"left-left", "right-right", "left-right", "right-left"}
    if mode not in valid_modes:
        raise ValueError(f"Invalid MATCH_MODE='{mode}'. Use one of: {sorted(valid_modes | {'both'})}")

    side0, side1 = mode.split("-")

    img0, mask0 = prepare_side_image_and_mask(
        inference0,
        terrain0,
        side0,
        image_source=image_source,
        section_obj=section0,
        flip_left_image=flip_left_image,
    )

    img1, mask1 = prepare_side_image_and_mask(
        inference1,
        terrain1,
        side1,
        image_source=image_source,
        section_obj=section1,
        flip_left_image=flip_left_image,
    )

    return img0, img1, mask0, mask1, side0, side1


def get_swaths_for_side(pings: list[pyxtf.XTFPingHeader], side: str) -> np.ndarray:
    """
    Returns the swath grid in the same horizontal orientation as the image used
    for matching.

    If FLIP_LEFT_INPUT_IMAGE=True, left swaths are flipped too. This keeps
    pixel-to-UTM conversion consistent with flipped left image coordinates.
    """
    swaths = extract_swaths_from_calculate_output(calculate_swath_positions(pings))

    side = side.lower().strip()
    mid = swaths.shape[1] // 2

    left_swath = swaths[:, :mid, :]
    right_swath = swaths[:, mid:, :]

    if FLIP_LEFT_INPUT_IMAGE:
        left_swath = np.fliplr(left_swath)

    left_swath = np.ascontiguousarray(left_swath)
    right_swath = np.ascontiguousarray(right_swath)

    if side == "left":
        return left_swath

    if side == "right":
        return right_swath

    if side == "both":
        return np.ascontiguousarray(np.hstack([left_swath, right_swath]))

    raise ValueError(f"Invalid swath side: {side}")


def mask_overlay_for_debug_local(img: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """
    White overlay = rejected pixels. Black/no overlay = pixels available for matching.
    """
    img_u8 = ensure_uint8_image(img)
    mask_u8 = ensure_gray(mask)

    if img_u8.ndim == 2:
        base = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)
    else:
        base = img_u8.copy()

    overlay = base.copy()
    overlay[mask_u8 > 0] = 255
    return cv2.addWeighted(base, 1.0 - alpha, overlay, alpha, 0.0)


def show_matching_inputs_debug(
    mode: str,
    inference0,
    inference1,
    terrain0,
    terrain1,
    image_source: str = "rho_gray",
    section0=None,
    section1=None,
    flip_left_image: bool | None = None,
):
    """
    Displays the exact image/mask arrays that will be passed to the matcher.
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    img0, img1, mask0, mask1, swath_side0, swath_side1 = build_matching_inputs(
        mode=mode,
        inference0=inference0,
        inference1=inference1,
        terrain0=terrain0,
        terrain1=terrain1,
        image_source=image_source,
        section0=section0,
        section1=section1,
        flip_left_image=flip_left_image,
    )

    print(f"Matching debug mode: {mode}")
    print(f"Image source: {image_source}")
    print(f"FLIP_LEFT_INPUT_IMAGE = {flip_left_image}")
    print(f"FLIP_LEFT_FINAL_MASK  = {FLIP_LEFT_FINAL_MASK}")
    print(f"Image 0 shape: {img0.shape} | Mask 0 shape: {mask0.shape} | Swath side: {swath_side0}")
    print(f"Image 1 shape: {img1.shape} | Mask 1 shape: {mask1.shape} | Swath side: {swath_side1}")
    print("Mask convention: black=keep, white=reject")

    overlay0 = mask_overlay_for_debug_local(img0, mask0)
    overlay1 = mask_overlay_for_debug_local(img1, mask1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0, 0].imshow(img0, cmap="gray")
    axes[0, 0].set_title("Image 0 passed to matcher")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(mask0, cmap="gray")
    axes[0, 1].set_title("Final mask 0 applied after matching")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(cv2.cvtColor(overlay0, cv2.COLOR_BGR2RGB))
    axes[0, 2].set_title("Mask 0 overlay on image 0")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(img1, cmap="gray")
    axes[1, 0].set_title("Image 1 passed to matcher")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(mask1, cmap="gray")
    axes[1, 1].set_title("Final mask 1 applied after matching")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(cv2.cvtColor(overlay1, cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title("Mask 1 overlay on image 1")
    axes[1, 2].axis("off")

    plt.suptitle("Exact matching inputs and final masks")
    plt.tight_layout()
    plt.show()

    return img0, img1, mask0, mask1


def run_parametrized_minima_registration(
    matcher,
    mode: str,
    inference0,
    inference1,
    terrain0,
    terrain1,
    pings0: list[pyxtf.XTFPingHeader],
    pings1: list[pyxtf.XTFPingHeader],
    image_source: str = "rho_gray",
    section0=None,
    section1=None,
    apply_mask: bool = True,
    ransac_reproj_threshold_pixel: float = 10.0,
    ransac_reproj_threshold_utm: float = 1.0,
    ransac_max_iters: int = 10_000,
    ransac_confidence: float = 0.995,
    ransac_outlier_percent: float = 0.5,
    flip_left_image: bool | None = None,
):
    """
    Local notebook registration runner that uses the same image/mask orientation
    as show_matching_inputs_debug().
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    if pings0 is None or pings1 is None:
        raise ValueError("pings0 and pings1 must always be passed for UTM conversion")

    image_source = image_source.lower().strip()

    img0, img1, mask0, mask1, swath_side0, swath_side1 = build_matching_inputs(
        mode=mode,
        inference0=inference0,
        inference1=inference1,
        terrain0=terrain0,
        terrain1=terrain1,
        image_source=image_source,
        section0=section0,
        section1=section1,
        flip_left_image=flip_left_image,
    )

    swaths0 = get_swaths_for_side(pings0, swath_side0)
    swaths1 = get_swaths_for_side(pings1, swath_side1)

    print(f"Matching mode: {mode}")
    print(f"Image source: {image_source}")
    print(f"Apply mask: {apply_mask}")
    print(f"FLIP_LEFT_INPUT_IMAGE = {flip_left_image}")
    print(f"FLIP_LEFT_FINAL_MASK  = {FLIP_LEFT_FINAL_MASK}")
    print(f"Image 0 shape: {img0.shape} | swath side: {swath_side0} | swath grid: {swaths0.shape}")
    print(f"Image 1 shape: {img1.shape} | swath side: {swath_side1} | swath grid: {swaths1.shape}")
    print(f"Mask 0 shape:  {mask0.shape}")
    print(f"Mask 1 shape:  {mask1.shape}")

    mkpts0, mkpts1, mconf = matcher.match_images(img0, img1)

    if apply_mask:
        mkpts0_kept, mkpts1_kept, mconf_kept, keep_mask = filter_matches_with_masks(
            mkpts0,
            mkpts1,
            mconf,
            mask0=mask0,
            mask1=mask1,
        )
    else:
        keep_mask = np.ones(len(mkpts0), dtype=bool)
        mkpts0_kept = mkpts0
        mkpts1_kept = mkpts1
        mconf_kept = mconf

    H_pixel, ransac_inliers_pixel = estimate_homography_ransac_custom(
        mkpts0_kept,
        mkpts1_kept,
        ransac_reproj_threshold=ransac_reproj_threshold_pixel,
        max_iters=ransac_max_iters,
        confidence=ransac_confidence,
        model="Euclidean",
        outlier_percent=ransac_outlier_percent,
    )

    mkpts0_utm = pixels_to_utm(mkpts0_kept, swaths0, img0.shape)
    mkpts1_utm = pixels_to_utm(mkpts1_kept, swaths1, img1.shape)

    H_utm, H_utm_local, mkpts0_utm_local, mkpts1_utm_local, utm_offset0, utm_offset1, ransac_inliers_utm = estimate_utm_homography_ransac(
        mkpts0_utm,
        mkpts1_utm,
        ransac_reproj_threshold=ransac_reproj_threshold_utm,
        max_iters=ransac_max_iters,
        confidence=ransac_confidence,
        outlier_percent=ransac_outlier_percent,
    )

    if H_utm is not None:
        H = H_utm
        ransac_inliers = ransac_inliers_utm
    else:
        H = H_pixel
        ransac_inliers = ransac_inliers_pixel

    tx_utm, ty_utm, theta_utm, theta_utm_deg = extract_euclidean_homography_params(H_utm)
    tx_utm_local, ty_utm_local, theta_utm_local, theta_utm_local_deg = extract_euclidean_homography_params(H_utm_local)

    num_ransac_inliers = 0 if ransac_inliers is None else int(np.sum(ransac_inliers))

    return RegistrationResult(
        mode=mode,
        image_source=image_source,

        img0=img0,
        img1=img1,
        mask0=mask0,
        mask1=mask1,

        mkpts0=mkpts0,
        mkpts1=mkpts1,
        mconf=mconf,
        keep_mask=keep_mask,

        mkpts0_kept=mkpts0_kept,
        mkpts1_kept=mkpts1_kept,
        mconf_kept=mconf_kept,

        mkpts0_utm=mkpts0_utm,
        mkpts1_utm=mkpts1_utm,
        mkpts0_utm_local=mkpts0_utm_local,
        mkpts1_utm_local=mkpts1_utm_local,

        H=H,
        H_pixel=H_pixel,
        H_utm=H_utm,
        H_utm_local=H_utm_local,

        tx_utm=tx_utm,
        ty_utm=ty_utm,
        theta_utm=theta_utm,
        theta_utm_deg=theta_utm_deg,

        tx_utm_local=tx_utm_local,
        ty_utm_local=ty_utm_local,
        theta_utm_local=theta_utm_local,
        theta_utm_local_deg=theta_utm_local_deg,

        ransac_inliers=ransac_inliers,
        ransac_inliers_pixel=ransac_inliers_pixel,
        ransac_inliers_utm=ransac_inliers_utm,

        utm_offset0=utm_offset0,
        utm_offset1=utm_offset1,

        num_matches=len(mkpts0),
        num_matches_after_mask=len(mkpts0_kept),
        num_ransac_inliers=num_ransac_inliers,
    )


In [ ]:
build_final_masks(
    section=section1,
    terrain=terrain1,
    output_mask_dir=output_dir + r"/section_01/masks",
    lower=lower1,
    upper=upper1,
    title="Final masks for section 1",
)

build_final_masks(
    section=section2,
    terrain=terrain2,
    output_mask_dir=output_dir + r"/section_02/masks",
    lower=lower2,
    upper=upper2,
    title="Final masks for section 2",
)

# 8) Debug the exact masks applied to matching

This plot shows the exact oriented image/mask pairs used by the matcher for the current `MATCH_MODE` and `IMAGE_SOURCE`.


In [ ]:
# Visualize the exact images and final masks that will be used by the selected MATCH_MODE.
# This calls build_matching_inputs(), so these are the same arrays passed to the matcher.
_ = show_matching_inputs_debug(
    mode=MATCH_MODE,
    inference0=inference1,
    inference1=inference2,
    terrain0=terrain1,
    terrain1=terrain2,
    image_source=IMAGE_SOURCE,
    section0=section1,
    section1=section2,
    flip_left_image=FLIP_LEFT_INPUT_IMAGE,
)


# 8) Multi-backend matcher definitions

Each matcher implements:

```python
match_images(img0, img1) -> mkpts0, mkpts1, mconf
```

This is the only interface required by `run_parametrized_minima_registration()`.

In [ ]:
def _ensure_gray_uint8_for_matching(img: np.ndarray) -> np.ndarray:
    img = ensure_uint8_image(img)

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return np.ascontiguousarray(img)


class LightGlueFeatureMatcher:
    """
    Direct LightGlue matcher using the cloned cvg/LightGlue repository.

    feature_type:
        "superpoint" -> SuperPoint + LightGlue
        "sift"       -> SIFT + LightGlue
    """

    def __init__(
        self,
        feature_type: str = "superpoint",
        device: Optional[torch.device] = None,
        max_num_keypoints: Optional[int] = 4096,
        resize=None,
        depth_confidence: float = 0.95,
        width_confidence: float = 0.99,
        filter_threshold: float = 0.1,
        mp: bool = False,
        flash: bool = True,
        compile_matcher: bool = False,
    ) -> None:
        from lightglue import LightGlue, SuperPoint, SIFT
        from lightglue.utils import rbd, numpy_image_to_torch

        self.feature_type = feature_type.lower().strip()
        self.device = device or torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.resize = resize
        self.rbd = rbd
        self.numpy_image_to_torch = numpy_image_to_torch

        if self.feature_type == "superpoint":
            self.extractor = SuperPoint(max_num_keypoints=max_num_keypoints).eval().to(self.device)
            lg_features = "superpoint"
        elif self.feature_type == "sift":
            self.extractor = SIFT(max_num_keypoints=max_num_keypoints).eval().to(self.device)
            lg_features = "sift"
        else:
            raise ValueError("feature_type must be 'superpoint' or 'sift'")

        self.matcher = LightGlue(
            features=lg_features,
            depth_confidence=depth_confidence,
            width_confidence=width_confidence,
            filter_threshold=filter_threshold,
            mp=mp,
            flash=flash,
        ).eval().to(self.device)

        if compile_matcher and hasattr(self.matcher, "compile"):
            self.matcher.compile(mode="reduce-overhead")

    def _to_lightglue_image(self, img: np.ndarray):
        img = _ensure_gray_uint8_for_matching(img)
        image = self.numpy_image_to_torch(img).to(self.device)
        return image

    def match_images(self, img0: np.ndarray, img1: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        image0 = self._to_lightglue_image(img0)
        image1 = self._to_lightglue_image(img1)

        with torch.inference_mode():
            feats0 = self.extractor.extract(image0, resize=self.resize)
            feats1 = self.extractor.extract(image1, resize=self.resize)
            matches01 = self.matcher({"image0": feats0, "image1": feats1})

        feats0, feats1, matches01 = [self.rbd(x) for x in [feats0, feats1, matches01]]

        matches = matches01["matches"]
        if matches.numel() == 0:
            return (
                np.empty((0, 2), dtype=np.float32),
                np.empty((0, 2), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            )

        mkpts0 = feats0["keypoints"][matches[..., 0]]
        mkpts1 = feats1["keypoints"][matches[..., 1]]

        if "scores" in matches01:
            mconf = matches01["scores"]
        elif "matching_scores0" in matches01:
            mconf = matches01["matching_scores0"][matches[..., 0]]
        else:
            mconf = torch.ones(matches.shape[0], device=matches.device, dtype=torch.float32)

        return (
            mkpts0.detach().cpu().numpy().astype(np.float32),
            mkpts1.detach().cpu().numpy().astype(np.float32),
            mconf.detach().cpu().numpy().reshape(-1).astype(np.float32),
        )


class SIFTMatcher:
    """
    Pure OpenCV SIFT + descriptor matching.

    This backend does not use LightGlue. It is useful as a classical baseline.
    """

    def __init__(
        self,
        nfeatures: int = 8000,
        n_octave_layers: int = 3,
        contrast_threshold: float = 0.01,
        edge_threshold: float = 10,
        sigma: float = 1.6,
        matcher_type: str = "flann",
        ratio_test: float = 0.75,
        cross_check: bool = False,
        flann_trees: int = 5,
        flann_checks: int = 64,
        enforce_unique_matches: bool = True,
        max_matches: Optional[int] = None,
    ) -> None:
        self.sift = cv2.SIFT_create(
            nfeatures=nfeatures,
            nOctaveLayers=n_octave_layers,
            contrastThreshold=contrast_threshold,
            edgeThreshold=edge_threshold,
            sigma=sigma,
        )
        self.matcher_type = matcher_type.lower().strip()
        self.ratio_test = ratio_test
        self.cross_check = cross_check
        self.flann_trees = flann_trees
        self.flann_checks = flann_checks
        self.enforce_unique_matches = enforce_unique_matches
        self.max_matches = max_matches

    @staticmethod
    def _unique_matches_by_distance(matches: list[cv2.DMatch]) -> list[cv2.DMatch]:
        # First keep the best match per query keypoint.
        best_by_query = {}
        for m in matches:
            old = best_by_query.get(m.queryIdx)
            if old is None or m.distance < old.distance:
                best_by_query[m.queryIdx] = m

        # Then keep the best match per train keypoint.
        best_by_train = {}
        for m in best_by_query.values():
            old = best_by_train.get(m.trainIdx)
            if old is None or m.distance < old.distance:
                best_by_train[m.trainIdx] = m

        return list(best_by_train.values())

    def _match_descriptors(self, desc0: np.ndarray, desc1: np.ndarray) -> list[cv2.DMatch]:
        if self.matcher_type == "bf":
            bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=self.cross_check)

            if self.cross_check:
                matches = bf.match(desc0, desc1)
            else:
                raw_matches = bf.knnMatch(desc0, desc1, k=2)
                matches = [m for m, n in raw_matches if m.distance < self.ratio_test * n.distance]

        elif self.matcher_type == "flann":
            index_params = dict(algorithm=1, trees=self.flann_trees)  # 1 = KDTree
            search_params = dict(checks=self.flann_checks)
            flann = cv2.FlannBasedMatcher(index_params, search_params)

            raw_matches = flann.knnMatch(desc0.astype(np.float32), desc1.astype(np.float32), k=2)
            matches = []
            for pair in raw_matches:
                if len(pair) < 2:
                    continue
                m, n = pair
                if m.distance < self.ratio_test * n.distance:
                    matches.append(m)
        else:
            raise ValueError("SIFT_MATCHER must be 'flann' or 'bf'")

        if self.enforce_unique_matches:
            matches = self._unique_matches_by_distance(matches)

        matches = sorted(matches, key=lambda m: m.distance)

        if self.max_matches is not None:
            matches = matches[: int(self.max_matches)]

        return matches

    def match_images(self, img0: np.ndarray, img1: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        gray0 = _ensure_gray_uint8_for_matching(img0)
        gray1 = _ensure_gray_uint8_for_matching(img1)

        kpts0, desc0 = self.sift.detectAndCompute(gray0, None)
        kpts1, desc1 = self.sift.detectAndCompute(gray1, None)

        if desc0 is None or desc1 is None or len(kpts0) == 0 or len(kpts1) == 0:
            return (
                np.empty((0, 2), dtype=np.float32),
                np.empty((0, 2), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            )

        matches = self._match_descriptors(desc0, desc1)

        if len(matches) == 0:
            return (
                np.empty((0, 2), dtype=np.float32),
                np.empty((0, 2), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            )

        mkpts0 = np.array([kpts0[m.queryIdx].pt for m in matches], dtype=np.float32)
        mkpts1 = np.array([kpts1[m.trainIdx].pt for m in matches], dtype=np.float32)

        distances = np.array([m.distance for m in matches], dtype=np.float32)
        mconf = 1.0 / (1.0 + distances)

        return mkpts0, mkpts1, mconf


def build_matcher(backend: str):
    backend = backend.lower().strip()

    if backend == "minima_sp_lg":
        return MinimaMatcher(
            method="sp_lg",
            ckpt=MINIMA_CKPT,
            use_path=False,
        )

    if backend == "lightglue_superpoint":
        return LightGlueFeatureMatcher(
            feature_type="superpoint",
            device=device,
            max_num_keypoints=LIGHTGLUE_MAX_NUM_KEYPOINTS,
            resize=LIGHTGLUE_RESIZE,
            depth_confidence=LIGHTGLUE_DEPTH_CONFIDENCE,
            width_confidence=LIGHTGLUE_WIDTH_CONFIDENCE,
            filter_threshold=LIGHTGLUE_FILTER_THRESHOLD,
            mp=LIGHTGLUE_MP,
            flash=LIGHTGLUE_FLASH,
            compile_matcher=LIGHTGLUE_COMPILE,
        )

    if backend == "lightglue_sift":
        return LightGlueFeatureMatcher(
            feature_type="sift",
            device=device,
            max_num_keypoints=LIGHTGLUE_MAX_NUM_KEYPOINTS,
            resize=LIGHTGLUE_RESIZE,
            depth_confidence=LIGHTGLUE_DEPTH_CONFIDENCE,
            width_confidence=LIGHTGLUE_WIDTH_CONFIDENCE,
            filter_threshold=LIGHTGLUE_FILTER_THRESHOLD,
            mp=LIGHTGLUE_MP,
            flash=LIGHTGLUE_FLASH,
            compile_matcher=LIGHTGLUE_COMPILE,
        )

    if backend == "sift":
        return SIFTMatcher(
            nfeatures=SIFT_NFEATURES,
            n_octave_layers=SIFT_N_OCTAVE_LAYERS,
            contrast_threshold=SIFT_CONTRAST_THRESHOLD,
            edge_threshold=SIFT_EDGE_THRESHOLD,
            sigma=SIFT_SIGMA,
            matcher_type=SIFT_MATCHER,
            ratio_test=SIFT_RATIO_TEST,
            cross_check=SIFT_CROSS_CHECK,
            flann_trees=SIFT_FLANN_TREES,
            flann_checks=SIFT_FLANN_CHECKS,
            enforce_unique_matches=SIFT_ENFORCE_UNIQUE_MATCHES,
            max_matches=SIFT_MAX_MATCHES,
        )

    raise ValueError(
        "Invalid MATCHER_BACKEND. Use one of: "
        "'minima_sp_lg', 'lightglue_superpoint', 'sift', 'lightglue_sift'."
    )

# 9) Run matching + masking + UTM conversion + custom RANSAC

The selected matcher only changes the source of `mkpts0`, `mkpts1`, and `mconf`. Everything after that is the same pipeline.

In [ ]:
pings1 = data_load[lower1:upper1]
pings2 = data_load[lower2:upper2]

matcher = build_matcher(MATCHER_BACKEND)
print("Matcher backend:", MATCHER_BACKEND)
print("Match mode:", MATCH_MODE)
print("Image source:", IMAGE_SOURCE)

result = run_parametrized_minima_registration(
    matcher=matcher,
    mode=MATCH_MODE,
    inference0=inference1,
    inference1=inference2,
    terrain0=terrain1,
    terrain1=terrain2,
    pings0=pings1,
    pings1=pings2,
    image_source=IMAGE_SOURCE,
    section0=section1,
    section1=section2,
    apply_mask=APPLY_MASK,
    ransac_reproj_threshold_pixel=RANSAC_REPROJ_THRESHOLD_PIXEL,
    ransac_reproj_threshold_utm=RANSAC_REPROJ_THRESHOLD_UTM,
    ransac_max_iters=RANSAC_MAX_ITERS,
    ransac_confidence=RANSAC_CONFIDENCE,
    ransac_outlier_percent=RANSAC_OUTLIER_PERCENT,
    flip_left_image=FLIP_LEFT_INPUT_IMAGE,
)

# Attach backend information for your own later checks.
result.matcher_backend = MATCHER_BACKEND

show_registration_result(result, max_draw=MAX_DRAW_MATCHES)

In [ ]:
print("Matcher backend:          ", result.matcher_backend)
print("Matching mode:            ", result.mode)
print("Image source:             ", result.image_source)
print("Raw matches:              ", result.num_matches)
print("After mask filtering:     ", result.num_matches_after_mask)
print("UTM RANSAC inliers:       ", 0 if result.ransac_inliers_utm is None else int(np.sum(result.ransac_inliers_utm)))
print("Pixel RANSAC inliers:     ", 0 if result.ransac_inliers_pixel is None else int(np.sum(result.ransac_inliers_pixel)))
print("tx_utm [m]:               ", result.tx_utm_local)
print("ty_utm [m]:               ", result.ty_utm_local)
print("theta_utm [deg]:          ", result.theta_utm_local_deg)

# 10) Optional debug: synthetic 180-degree self-rotation

This uses the selected matcher backend too. It is useful to compare whether each matcher can recover a strong self-match under a known synthetic transformation.

In [ ]:
if RUN_DEBUG_SELF_ROTATION:
    debug_result = run_debug_self_rotation_registration(
        matcher=matcher,
        mode=MATCH_MODE,
        inference_obj=inference1,
        terrain_obj=terrain1,
        pings=pings1,
        image_source=IMAGE_SOURCE,
        section_obj=section1,
        apply_mask=APPLY_MASK,
        ransac_reproj_threshold_pixel=RANSAC_REPROJ_THRESHOLD_PIXEL,
        ransac_reproj_threshold_utm=RANSAC_REPROJ_THRESHOLD_UTM,
        ransac_max_iters=RANSAC_MAX_ITERS,
        ransac_confidence=RANSAC_CONFIDENCE,
        ransac_outlier_percent=RANSAC_OUTLIER_PERCENT,
    )

    debug_result.matcher_backend = MATCHER_BACKEND
    show_registration_result(debug_result, max_draw=MAX_DRAW_MATCHES)
else:
    print("RUN_DEBUG_SELF_ROTATION=False, skipping debug self-rotation test.")

# 11) Backend tuning notes

## Recommended first comparisons

Run the same timestamps and same `MATCH_MODE` with:

```python
MATCHER_BACKEND = "minima_sp_lg"
MATCHER_BACKEND = "lightglue_superpoint"
MATCHER_BACKEND = "sift"
```

Then compare:

- `Raw matches`
- `After mask filtering`
- `UTM RANSAC inliers`
- `theta_utm_deg`
- visual match quality after RANSAC

## Practical defaults for sonar waterfalls

For `lightglue_superpoint`, start with:

```python
LIGHTGLUE_MAX_NUM_KEYPOINTS = 4096
LIGHTGLUE_RESIZE = None
LIGHTGLUE_FILTER_THRESHOLD = 0.1
```

For `sift`, start with:

```python
SIFT_NFEATURES = 8000
SIFT_CONTRAST_THRESHOLD = 0.01
SIFT_RATIO_TEST = 0.75
SIFT_ENFORCE_UNIQUE_MATCHES = True
```

If SIFT gives too many weak matches, increase `SIFT_RATIO_TEST` strictness by lowering it, for example `0.65`. If it gives too few matches, try `0.80` or increase `SIFT_NFEATURES`.

If LightGlue gives too many weak matches, increase `LIGHTGLUE_FILTER_THRESHOLD`. If it gives too few matches, lower it or increase `LIGHTGLUE_MAX_NUM_KEYPOINTS`.